# Local-evidence token profiles

This notebook compares existing local-evidence and SemEval random-forest scores projected onto token positions. It does not train models or run language-model inference.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from notebooks.utils.local_evidence_comparison_utils import (
    plot_comparison_token_profiles,
    select_configured_comparison_texts,
)
from notebooks.utils.semeval_random_forest_token_visualization_utils import (
    MODEL_FAMILY,
    SCORER_LLM as SEMEVAL_SCORER_MODEL,
    WINDOW_LENGTHS,
    build_gold_boundaries,
    load_selected_annotations,
    load_selected_token_labels,
    load_selected_windows,
    load_visualization_models,
    resolve_prepared_inputs,
    scan_annotation_ids,
    scan_stage1_ids,
    score_visualization_windows,
    select_eligible_text_ids,
)

DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
SCORER_MODEL = "gemma_3_270m_it"
CLASSIFIERS = ("logistic_regression",)
TRAINING_WINDOWS = (32, 64)
CONFIGURATIONS = tuple((model, window) for model in CLASSIFIERS for window in TRAINING_WINDOWS)
TEXTS_PER_GROUP = 3
SEED = 42
SMOOTHING_WINDOW = 10
PARTITION = "test"
SEMEVAL_ARTIFACT_RUN_ID = "feature-vector-classification-2026-09-21-logreg-random-forest"
SEMEVAL_DATASET = "semeval2024_task8_subtask_c_train"
SEMEVAL_SPLIT = "MG"
SEMEVAL_TEXT_COUNT = 20
SEMEVAL_SELECTION_SEED = 42
SEMEVAL_CONFIGURATIONS = tuple((MODEL_FAMILY, window_length) for window_length in WINDOW_LENGTHS)
SEMEVAL_ANNOTATION_PATH = (
    DATA_ROOT
    / "raw/semevals_2024-task8/SemEval2024-task8/subtaskC/SemEval2024_task8_C_train_MG.csv"
)
DEFAULT_SCORED_WINDOWS_PATH = (
    DATA_ROOT / "experiment-results/local-evidence-classification" /
    f"mixed-bbc-all-llms-2026-09-10-v2-{SCORER_MODEL}/csv_exports/scored_windows.csv"
)
CONFIGURATION_PATHS = {
    configuration: Path(os.environ.get(
        f"LOCAL_EVIDENCE_{configuration[0].upper()}_{configuration[1]}_PATH",
        DEFAULT_SCORED_WINDOWS_PATH,
    ))
    for configuration in CONFIGURATIONS
}
GROUPS = {
    "Kaggle HG": {"dataset": "kaggle_train", "split": "HG"},
    "Kaggle MG": {"dataset": "kaggle_train", "split": "MG"},
    "MAGE HG": {"dataset": "mage_train", "split": "HG"},
    "MAGE MG": {"dataset": "mage_train", "split": "MG"},
    "Outfox HG": {"dataset": "outfox_chatgpt_train", "split": "HG"},
    "Outfox MG": {"dataset": "outfox_chatgpt_train", "split": "MG"},
    "BBC HG": {"dataset": "bbc_news_content", "split": "HG"},
}
SELECTED_TEXT_IDS = {
    # "Kaggle HG": ["kaggle_train::HG::910", "kaggle_train::HG::1245"],
}


In [ ]:
def load_configuration_frames() -> dict[tuple[str, int], pd.DataFrame]:
    frames = {}
    loaded_paths = {}
    columns = ["partition", "dataset", "split", "text_id", "start_token", "stop_token", "mg_score", "model", "window_length"]
    for configuration, path in CONFIGURATION_PATHS.items():
        if not path.is_file():
            raise FileNotFoundError(f"Missing scored-window artifact for {configuration}: {path}")
        if path not in loaded_paths:
            loaded_paths[path] = (
                pd.read_csv(path, usecols=lambda column: column in columns)
                if path.suffix.lower() == ".csv"
                else pd.read_parquet(path, columns=columns)
            )
        frame = loaded_paths[path]
        required = {"partition", "dataset", "split", "text_id", "start_token", "stop_token", "mg_score"}
        missing = required.difference(frame.columns)
        if missing:
            raise ValueError(f"{path} is missing columns: {sorted(missing)}")
        model, window_length = configuration
        if "model" in frame:
            frame = frame.loc[frame["model"].eq(model)].copy()
        if "window_length" in frame:
            frame = frame.loc[frame["window_length"].eq(window_length)].copy()
        frames[configuration] = frame.assign(model=model, window_length=window_length)
    return frames

configuration_frames = load_configuration_frames()

def plot_group(group_name: str, group_filter: dict[str, object]) -> None:
    frames = {
        configuration: frame.loc[
            frame["partition"].eq(PARTITION)
            & frame[list(group_filter)].eq(pd.Series(group_filter)).all(axis=1)
        ].copy()
        for configuration, frame in configuration_frames.items()
    }
    selected = select_configured_comparison_texts(
        frames,
        count=TEXTS_PER_GROUP,
        seed=SEED,
        partition=PARTITION,
        configurations=CONFIGURATIONS,
        selected_text_ids=SELECTED_TEXT_IDS.get(group_name),
    )
    display(selected.assign(group=group_name))
    for row in selected.itertuples(index=False):
        scored = pd.concat(
            [frame.loc[frame["split"].eq(row.split) & frame["text_id"].eq(row.text_id)] for frame in frames.values()],
            ignore_index=True,
        )
        token_count = int(scored["stop_token"].max())
        figure = plot_comparison_token_profiles(
            text_id=row.text_id,
            token_count=token_count,
            scored_windows=scored,
            title=f"{group_name} / {SCORER_MODEL}",
            smoothing_window=SMOOTHING_WINDOW,
            configurations=CONFIGURATIONS,
        )
        display(figure)
        plt.close(figure)

available_groups = {
    group_name: group_filter
    for group_name, group_filter in GROUPS.items()
    if any(
        not frame.loc[
            frame["partition"].eq(PARTITION)
            & frame[list(group_filter)].eq(pd.Series(group_filter)).all(axis=1)
        ].empty
        for frame in configuration_frames.values()
    )
}
for group_name in GROUPS:
    if group_name not in available_groups:
        display(Markdown(f"**Warning:** {group_name} has no scored-window rows and was skipped."))
for group_name, group_filter in available_groups.items():
    plot_group(group_name, group_filter)

def plot_semeval_profiles() -> None:
    models = load_visualization_models(DATA_ROOT, run_id=SEMEVAL_ARTIFACT_RUN_ID)
    prepared = resolve_prepared_inputs(DATA_ROOT, SEMEVAL_DATASET, models, split=SEMEVAL_SPLIT)
    selected = select_eligible_text_ids(
        {window_length: path for window_length, (path, _) in prepared.items()},
        stage1_ids=scan_stage1_ids(
            DATA_ROOT / "processed", SEMEVAL_DATASET, SEMEVAL_SCORER_MODEL, SEMEVAL_SPLIT
        ),
        annotation_ids=scan_annotation_ids(SEMEVAL_ANNOTATION_PATH),
        count=SEMEVAL_TEXT_COUNT,
        seed=SEMEVAL_SELECTION_SEED,
    )
    selected_ids = set(selected["text_id"].astype("int64"))
    windows_by_length = load_selected_windows(prepared, selected_ids)
    scored_windows = score_visualization_windows(models, windows_by_length)
    token_labels = load_selected_token_labels(
        DATA_ROOT / "processed", SEMEVAL_DATASET, SEMEVAL_SCORER_MODEL, SEMEVAL_SPLIT, selected_ids
    )
    annotations = load_selected_annotations(SEMEVAL_ANNOTATION_PATH, selected_ids)
    gold_boundaries = build_gold_boundaries(annotations, token_labels).set_index("text_id")

    display(Markdown("## SemEval MG random-forest token profiles"))
    display(selected)
    for text_id in selected["text_id"].astype("int64"):
        figure = plot_comparison_token_profiles(
            text_id=text_id,
            token_count=len(token_labels[text_id]),
            scored_windows=scored_windows.loc[scored_windows["text_id"].eq(text_id)],
            title=f"SemEval MG / {SEMEVAL_SCORER_MODEL}",
            smoothing_window=SMOOTHING_WINDOW,
            configurations=SEMEVAL_CONFIGURATIONS,
            gold_token_boundary=int(gold_boundaries.loc[text_id, "gold_token_boundary"]),
        )
        display(figure)
        plt.close(figure)


plot_semeval_profiles()
